# DDL — the five schemas

One notebook, five schemas. This is **deployment, not pipeline**: it declares where things
live and is run once, by hand. It is not a task on the monthly Workflow, because re-running
`CREATE SCHEMA IF NOT EXISTS` every month is work that can only ever do nothing.

| Layer | Job | Write |
|---|---|---|
| `landing` | data exactly as the source sent it — no transformation of any kind | OVERWRITE |
| `bronze` | the same rows under one contract: every column STRING, nothing rejected | OVERWRITE |
| `silver` | all quality work — scale repair, the study window, total return | MERGE |
| `gold` | the dimensional model, SCD2 built via MERGE | MERGE |
| `semantic` | thin views over Gold. No tables, no new arithmetic | views |

Catalog name has hyphens, so every reference needs backticks.

In [0]:
CREATE SCHEMA IF NOT EXISTS `index-vs-trust-pipeline`.landing
COMMENT 'Sources exactly as they arrived';

In [0]:
CREATE SCHEMA IF NOT EXISTS `index-vs-trust-pipeline`.bronze
COMMENT 'Landing rows under one contract: every column STRING';

In [0]:
-- Silver is the first layer allowed to change a value, so it also keeps the audit trail.
CREATE SCHEMA IF NOT EXISTS `index-vs-trust-pipeline`.silver
COMMENT 'Cleaned monthly return series, and the audit trail for every value changed';

In [0]:
-- A one fact and four dimensions; dim_manager reaches the fact through a bridge.
CREATE SCHEMA IF NOT EXISTS `index-vs-trust-pipeline`.gold
COMMENT 'Dimensional model: dim_date, dim_ticker (SCD2), and the two performance facts';

In [0]:
-- No arithmetic here, so every dashboard figure traces back to a Gold table.
CREATE SCHEMA IF NOT EXISTS `index-vs-trust-pipeline`.semantic
COMMENT 'Thin views over Gold for the dashboard. No tables, no arithmetic';

## Verification

Expected: five rows — `bronze`, `gold`, `landing`, `semantic`, `silver`.

In [0]:
SHOW SCHEMAS IN `index-vs-trust-pipeline`;